# **가격 추이 정보**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 전체 CSV 로딩
volume_path = "/Volumes/bronze_api/agrofood_pricesequel/volumn/*.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(volume_path)

# 가격 관련 컬럼 숫자형으로 변환
price_cols = [c for c in df.columns if 'prc' in c]
numeric_cols = ['ctgry_cd', 'grd_cd', 'item_cd', 'se_cd', 'unit_sz', 'vrty_cd'] + price_cols

df2 = df
for col_name in numeric_cols:
    df2 = df2.withColumn(col_name, F.col(col_name).cast(DoubleType()))

print(f"총 행 수: {df2.count():,}")
print(f"컬럼 수: {len(df2.columns)}")
print()

# 기초통계 (수치형)
display(df2.select(price_cols).describe())

In [0]:
# 컬럼 한글명 매핑
col_kor_map = {
    "ctgry_cd": "부류코드",
    "ctgry_nm": "부류명",
    "item_cd": "품목코드",
    "item_nm": "품목명",
    "vrty_cd": "품종코드",
    "vrty_nm": "품종명",
    "grd_cd": "등급코드",
    "grd_nm": "등급명",
    "se_cd": "구분코드",
    "se_nm": "구분명",
    "unit": "단위",
    "unit_sz": "단위크기",
    "exmn_ymd": "조사일자",
    "prc": "가격",
    "year": "연도",
    "year_month": "연월"
}

total = df2.count()
notnull_counts = df2.select(
    [F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c) for c in df2.columns]
).toPandas().T
notnull_counts.columns = ['notnull_count']
notnull_counts['notnull_pct'] = (notnull_counts['notnull_count'] / total * 100).round(2)
notnull_counts['col_name'] = notnull_counts.index
notnull_counts['col_kor'] = notnull_counts['col_name'].map(col_kor_map).fillna("")
notnull_counts = notnull_counts[['col_name', 'col_kor', 'notnull_count', 'notnull_pct']]
display(notnull_counts)

In [0]:
# 컬럼 한글명 매핑
col_kor_map = {
    "ctgry_cd": "부류코드",
    "ctgry_nm": "부류명",
    "item_cd": "품목코드",
    "item_nm": "품목명",
    "vrty_cd": "품종코드",
    "vrty_nm": "품종명",
    "grd_cd": "등급코드",
    "grd_nm": "등급명",
    "se_cd": "구분코드",
    "se_nm": "구분명",
    "unit": "단위",
    "unit_sz": "단위크기",
    "exmn_ymd": "조사일자",
    "prc": "가격",
    "year": "연도",
    "year_month": "연월"
}

total = df2.count()
null_counts = df2.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df2.columns]
).toPandas().T
null_counts.columns = ['null_count']
null_counts['null_pct'] = (null_counts['null_count'] / total * 100).round(2)
null_counts['col_name'] = null_counts.index
null_counts['col_kor'] = null_counts['col_name'].map(col_kor_map).fillna("")
null_counts = null_counts[['col_name', 'col_kor', 'null_count', 'null_pct']]
display(null_counts)

In [0]:
for c in df2.columns:
    null_rows = df2.filter(F.col(c).isNull())
    if null_rows.count() > 0:
        print(f"\n컬럼: {c} (null값 {null_rows.count()}개)")
        display(null_rows)

In [0]:
cat_cols = ['ctgry_nm', 'item_nm', 'vrty_nm', 'grd_nm', 'se_nm', 'unit']
print("범주형 컬럼 고유값 수")
for c in cat_cols:
    cnt = df2.select(c).distinct().count()
    print(f"  {c}: {cnt}개")

display(df2.groupBy('ctgry_cd','ctgry_nm').count().orderBy(F.desc('count'))) # 부류명
display(df2.groupBy('item_cd','item_nm').count().orderBy(F.desc('count')))  # 품목명
display(df2.groupBy('vrty_cd','vrty_nm').count().orderBy(F.desc('count')))  # 품종명
display(df2.groupBy('grd_cd','grd_nm').count().orderBy(F.desc('count')))   # 등급명
display(df2.groupBy('se_cd','se_nm').count().orderBy(F.desc('count')))    # 구분명
display(df2.groupBy('unit').count().orderBy(F.desc('count')))     # 단위

In [0]:
date_col = "exmn_ymd"

# 문자열 -> 날짜형 변환
df2 = df.withColumn(
    date_col,
    F.to_date(F.col(date_col).cast("string"), "yyyyMMdd")
)

# 연도, 연월 컬럼 추가
df2 = (
    df2
    .withColumn("year", F.year(F.col(date_col)))
    .withColumn("year_month", F.date_format(F.col(date_col), "yyyy-MM"))
)

display(df2.select(date_col, "year", "year_month").limit(5))

In [0]:
combo_count_name = (
    df2.groupBy("ctgry_cd", "ctgry_nm", "item_cd", "item_nm", "vrty_cd","vrty_nm",'grd_cd','grd_nm','se_cd','se_nm','unit')
    .count()
    .orderBy(F.desc("count"))
)
# 
display(combo_count_name)